# DDPM Part 2 of 3: the training loss

Part 2 of 3, after **u18-p04** (you may copy your `q_sample` from there into this file). Compute the DDPM simple loss for one batch:

$$\mathcal{L} = \operatorname{mean}\Big(\big(\varepsilon_\theta(x_t, t) - \varepsilon\big)^2\Big), \qquad x_t = \sqrt{\bar\alpha_t}\,x_0 + \sqrt{1 - \bar\alpha_t}\,\varepsilon.$$

```python
def ddpm_loss(model, x0, betas, generator):
```

- `model(x_t, t)` takes a batch `x_t` shaped like `x0` and a long tensor `t` of shape `(B,)` with zero-based steps, and returns the predicted noise with the shape of `x0`.
- `x0`: a float tensor `(B, ...)`. `betas`: float64 tensor `(T,)`; `alpha_bars[i]` is the product of `1 - betas[0..i]`.
- `generator`: a `torch.Generator`. Use it for **all** randomness, in exactly this order:

```python
t = torch.randint(0, T, (B,), generator=generator)
eps = torch.randn(x0.shape, generator=generator, dtype=x0.dtype)
```

Return the mean over **all** elements of the squared difference, a 0-dimensional tensor that is differentiable with respect to the model's parameters. Do not modify `x0`.

**Example.** For `model = lambda x, t: 0.5 * x`, `x0 = torch.ones(2, 3)`, `betas = torch.linspace(0.1, 0.3, 3, dtype=torch.float64)` and `torch.Generator().manual_seed(0)`, the drawn steps are `t = [2, 0]` and the loss is about `0.6340`.

**Restrictions.** Only `torch` (with `torch.nn.functional`).

**Constraints.** $B \le 256$, `x0` up to `(256, 3, 32, 32)`, $T \le 1000$. Time limit 1 second with a small model.


## How to use this notebook

1. Write your solution in the next cell. Running it saves the file `problem5_submission.py`, the same way the contest's Colab notebook does.
2. Run the grader cell at the bottom. It runs the example and the hidden tests and checks the import and ban rules.
3. Edit, re-run the solution cell, then re-run the grader.

From the USAAIO Prep study site (https://bengreff.github.io/usaaio-prep/); not affiliated with or endorsed by USAAIO.

In [ ]:
%%writefile problem5_submission.py
import torch
import torch.nn.functional as F


def ddpm_loss(model, x0, betas, generator):
    # Your code
    return loss


Optional: import it the way the grader does and try the example from the statement.

In [ ]:
import importlib
import problem5_submission as submission
importlib.reload(submission)
# e.g. submission.ddpm_loss(...)

## Grade

In [ ]:
#@title Grader: run this cell to grade your solution { display-mode: "form" }
# The grading code is hidden on purpose (double-click to see it). It writes the site's test harness and this
# problem's grader to a temporary folder, imports your file the way the contest grader does, checks the import
# and ban rules, runs every test (including hidden ones) and prints a table.
import ast, base64, contextlib, importlib.util, io, json, os, signal, sys, time, traceback, zlib
_P = json.loads(zlib.decompress(base64.b64decode("eNq9Wwlz2ziW/itYddWEzMiMrBzbo7R7N4e7J1u5KslUz5TlYkMiJHNMgWyCiu32+L/v9x4AEpRkJU7PblJliSDwLrwb0PUgzwYTMVgffn9QjR4PhmKwUo3E0PX2myZvCkVjL1++fyPey7oRY1EuxMOJaM6UaGqZ61wvRVEaQ/MLZUypPZBi9IjB53rdKIPRwzEes3yxyOfrornCyEMMKN3U9H2QZdUq9aBWZbYuVKrligmo6nJWqNXj1Kxnq9yYHGgwSxZFeaGyNF9VZd0QjpNBU9bzs8Ep3s6k1op4OnFPgF5WNGshC6OIQWWatMlXqlw3Kb0Yj2hULi2opdIyT4jitcdoh3SZG5VWtcryeePeMN6kljrLdcP4Tbmu50y+zGTVqOxgPBo/OfgwPnj/kFZUEOgYr5t6rW7wvMgLltP14EzWGrJMKhbMlP//VRWVqg0kLxvh5CGWtcxo0ApARL8u6nIl3HI/ev/XOJnqqX7mpovciHmpG0j+wVrnjXnwA338+MBBxXOVZz8+sLNBRCJeNSJTixxQBclMLNaaGTeCdijjwfS+pa2R54r0Y6q7zRJ2P8WvGPo1Ecd/f/bm/evjj6LITWOsMhFYAmaEOSsvtMCiD2stos+5yUFV/FR8JHCNqNdAqz6r+ooXgbWfZF6sa6yEQhol6FFc5M2ZkALwjFwqcVHnDRgWi7IWz5WeCCOvxAWReyEN3pZ6ORSaoIpsvarEWZ5lmJ7rat0Ykp7dBe1EOi+rq/aBFIjk6x411l8JQNUVjX4nXpSFnIkX7/9mhKyVMKSzNclKEyn3jHgzFu/rcihMCVOay4Igwq6MWK0hlJkioUJEJVlOjtdFTnKYgf8zbOVCzqF7CWEaJyPaXCmWa9r/yGI2gFWog+asVjITplLYMMyqy/XyrLiCthQLAVpAxbysSdC1IqE3Il8QgXhk63J8OHMhoXx8f3z8Mv3p2YtP7z6II8JODE/1vJDATtsQHV/OVUWaEk+mWuBfhVd2FhQKEldZBF3MhmJlln4O8OqShKwzN0L/us2NaG4HJNVVdOnXwpkEa/wOkW0Gowvwn2vTSD1X0eXQvk8+KW3KOg6WM1rVrGstLpMMfnJ+FsXJvFrjL29zFNvJitkUrxjdcV2XdQDFshzA0lUijaxreQW6Oz7MmaxUtCybIZRS01/o59F0AM2Gv5wOPGUGsm7WVaEi4hzz44SXOlpYppjjJxGseCgW08E1AbzBhhuLS1wb+B11Wak5vJO4DubfAF1H2RxueR9lQ1E3ZXF0qA6eDIV0X7/39MKyLkByS6w0zRUYXRSlJMJonHH2XwTMLC1/xNKF/XoLO5YBNz3exZt/xey1CLAhuYF7yxusBh1FEcUCnoK0MHx34d6F6MmTIg6SJ9S0JteLFrhTZNrvonBChDCcuOiPExf9CRVP1TUkxoIg4lbykj7kzERLcSAu4jgm2GA0/10JhWAmRmR8O0ylLyd2dOKzLOAfRFTIekluV5HCimt8TJKHy5u4t/fqt7UsvkIr+0ZlpxLROmNNH8YsURAhm6Z276cD3o4OyA51CRSkm/N/rhahajD9qZMDiNsyplCqLQTalckmxVAGopaZ72DAGV/j1X/UPdJoFoZ6m5GblE2Et4O+mVv3gxEGO9It2aKfRiEKBzhOUs670rRHzhLuj3bOYb3nJ93zUPuUpkbLCpG8+f91zUWp1R2dMlB6rbwkleTYPogREjO23j49EaV8Q05c4pCePnM96ghglClVHX1CphcomCPwE8S3Sd42gLgXQmgoIZj87rIneGxLJIdi9u1yl18ldzvF2gXj+1axSxI7wzF9V+AQSYvERLN4l6/B8s7P7Fze+omZ+8p727ds4mBovR00W4d75eEwhCB5KdPVGkaB7CZa6KG4D3cKy7h///yCvnlKbOr4AhFALDRjXlCCSjwgx0EWuIR5SY2UsV6uV0jKEUJEVci5SnzWyVGf7Al+8aSzLBlzPitpAWE8tTORnVHw0NEmQfY1LcmhyGDYxLRUIZVRtUR8+z2vIruGcfQUnP1Jq1zOiVyhxIEuZooS/HyRU2ZJKXPHy3fXufizOLx5Ki7K+pyyesnKK2j7kI+2/tIJGcR3EqYkPteRUZQKgqqvlfLTUMTwrmHKvSOT5jIS8dAg9wYHJCDOde+1K5G82xqKUL0DiJpKMBRIK0MscR7+Rs7FHAhoCZi0WTdVK5zYU3KA6uJgts7gSWmYc3Ck5TkVUOsK5iSai3yuCCAklCNClwtnwaDZS9SwCtHIAvkShW+CgBVrvEvEJ3qR1xgGM8U9QkFxwSIjCausp1bNiLJJQEhQXS5SBqNq722+oEpZs3sxcpTGpSOO3yPhNlE8EGHZ0KlkSqpYkzVE41DvsIlA88ORg7Thimaoac67ITkeivMxsPX9I5M83BjscfJlUTCdJIgxi2EcDLMYULFFGYXYW8QR5pshQ2xHTVmy3qD4z5pJMl7cIMuHlqmnrupqlZiVCWbHgMRaUz1/7UTj1nENicLOb/GOf4wSlSXK4AVUa0kJrC0yr90+MSCyjr0GWuslrFNlfsO6AocaIeUK0l5IqF/aTpxys2RtpMzLlDsMvS7HzzQCCwBlXfMgZ6dxlhcZWd2cOiMomHOyBZr5Px/fvQWtmi3HNBkoZFutrpozjBy8Eh6P+MG1ONIsr38UP3Q48BLPsMp/8Vjz41bJD2MLyn9s7GVT5LN2yH5gJFk3edENl+3Xf5qyaxJUsjkLl5t8qWW3zFyZjfaCf6gRF2Zyft6OQIupCWdsr8HMc7CpVevuMHCeNweFwjRxmDxJDl1OZybUJ1pRz27ILgXVhQs+3vlVMoeUl1PtcSSLvIBG+8doOgDZZa2oAnSdFmSiyf2/6XNNPRxTFtRRKbkDAIR5RbuW3HeZ4l+PPxzDcpwskvf4jNKU2mBpGidwXLQcWV0la1A21ZBKQpMTaKuqm2gEY0D6QGBiANzV++JekPgOsfo3ORHHj0Zj7oS9/uXZPz6mz5+9fXv8EhRcTwepayKmKTEDxvx+2kd1qebuG/J8+21erirQOh3cTPXrd+/ep2/fvTz+CHARdCX5qYSt0pdfoLjKfn1mrvS8ffEaOeQLwLBPH1Xw8BJJZvf0s9IUn8v6+LKKwxbLJ9uL+VKXJZWo9FYR6dh6BX+DsKVai+VS0QMKuxGU2czT+Zman5vI1HPa4kb24+0Ha++UDVNjGHIsz61wVmYJwTyFW6QM0Pb42GxMc6/dHU6GXCuRAlVT236q6YUoG4ROTndls02tkNixlKAlqK9BaD8P/XiF+vyS81Bqyqnt/PDEkT2BtnBf2FE/ITdpeLkrkUtt/cy1SuhTlzcT+o7ZKH5uHIGuKU0hAfJK4JhhJxudapLRyakjdCbbcY6TjQ853A5N14aBBeMz2ENqe6VunFQcKSNsN89gKpSJ1TZRsIABwc6PqKKhpDufUVce7Jxz7qaKhBTbIeA+eSDzlWxW6yIcWXB3BK6afDOEfyGL84g2YyN0B2k6Tbf6bEuDzaqiTWVpYsLkTrYDmGupIF+OJE+ipHxFrQX7mAAbBEmZY7RC5gl3RMUcAV8xsXYj4snu2BjsRSKzzOGItyd3exPMS0xV5LTdhPPk4PB0x8Ju85J1lVHWvWNtWCQW+8X4E7zeJjfYUOwVy9E13sF+Z1A7ZMkrnCDx/Q9IcVOCgLYhhUB0PQns0oCNpbcJ75uWMp93FfozbzubvG8oBNNBhvZVUN8GHnkfwDy7navdk3Zh7ILVJk42/ERWldJZ5Ds+dtF0YF2ede//1ZfXLiwcBZ/n+p2LYs/Wy2eGYlBseysbC5LSzXsjmzdIGbd02vqgOxIHwgKFDEBSkuhA3er+fRZgrcSqPI2yQdC85J8llafsYKMATxzv9i5/pogCwXhQ1/eG4p4F4i2KDPAe8KHuXd6jVuzN7n7iLvqpZbGDfONp74B9154gzqQ2JNv5mYsQnAp2AYLbz0qiiP07f8ZPbTVaXMgrc2CPWFHJIC/MW0CklA5aqYuroceJyqULNZHVAttEcqkUJR1XolBLlOYXlDYJnpkkSRwEMyRlIitpUyzgs5zrS7sLURA0/8TBMYjCllwXfKGG/xJREEv/JHp5YaBCwHBn1QEFRjj5EPzIxDt0B5DjP7LDuvQ42iPZbpdBeZ/51BpRWlY2n+x322i6NbIN0/sahqmihF78t/CwqWQN7EZENnNyGE5Gp6Fy72D/LiLosO5jnp3bFss8+m0M2zSJlb3HKIbJjEmpa3WmNBexbe7Iq/69/DvObvZU6os6B5jiKuVMNgIIFLopVVP9jP6dVgdMZ6bMvM65qhAVVLWhmx7uJJon5P74mgo2mKuieYYP6/sdphm1VXzdmqBuxvcmbWaRStK0fYGSzyWbBByZ5oKD+oLwAAR2akHlp/I5H1V+LQsuK72glgk1rMjLcm/SKR3+UjoWZO29IdoKVhiazcdpHflOmHRiwickKjgfoez/5pqx3tB8L+wVMvGorcXgaTerXKpkZb38fNKmiJ6VW2eOT4OS2EmKEvAj0U556NN16D3GqeGQFKXMTBQxEQ9IbcgmuBUxIHiIWdTJiLyzQ/lEEB0x4QQvDe4mHoltu7AF49Tdt5ny5RrTPdrGjFVZV0mTJtLA2xJbEMI/6aCdEjm3lKKtoVMGSyZoTixVpxwt6Imt5DRMZ31J5/NZw33HLbShl+CuRcTipMshJrLTwyhvtcS3PBcgOi+Tjw21T16988KzTZ7EfkTu6eOrn5+9/vBm6Kp0NxXuLOXmrj8FDnxZ/9YSyXI88rT0q2O+BNM1qxK6sFQrGJ/djAiUbqZZnkbV5ISi9mS++vTqzfGH9MPxs9fDlrqNNMdUirSn3wpLaDSl3ozt7FB3ko9MAo6C215WNbrGHLYpcFUb+NazbXQOFiMk1BH92VwIi7HzzAm95oLh1Cr+NktsQ6pOqAeU2mXkeeJvENyo35twbZdeS8IpojOOU+vNLI/kgdmrscdtyvKczxTEtd+PSTKiDrB3XQ7Lc2lU2yGyPZCuJfb89fFodNgRcCc2bqPYHQwxmXQSQzfESrdRLgnaE49as94CvUtQ3q8QXugzaRSfxEcowB+NRqPJ6TcbMl9N7PdxeCi832Hn+LNif6IJXob2Vfzlze1klZXKJk32sh2iC4G4Cev2O7KwvJtNTgf20h9ZYRsy2nuALd/uHuGXbW8ZGN/yNluy0Nwsd93PNbZ8lWenkGfwVwd9Eu/cHsUaShoiFCoL69jpG/n2z7I2HkeSN2plIr49o/tRgb2qP/enAxQJMqJFHDa84CXoEJJTEsK4kTxzOCbtr9fwW1YpKFvhWwTas7bhcMk953qtwv2b2wBr/eGEgZw8npzCf/O5MKh1XWrB7nE6cKD95AAbvaVuMJU9u8oU52/toQ8NjZLRzV3OxLYvPnxD1Pmjkccdzm175W9yaW4TTjrJkaVS4r19d2PLg3eLbe5hbZyddVHqpT+X2fTZIsq1vWXG2fyQCoiFrIU/GYxDJ+CQ8+HGRk/7DzLc0kzp9Y67KneKJf8uWsJ7BnxekX1lDOlgtQpO8GpocRbddlA7FA93hTeXzp76gqxuHdudQ1G7wOaqR1Q6Rl3uui8rZacyK8si2iLMviNQTU9zCVzTA9eu8AXKrSGFrzUvhC94rHdLU6pv6KBs4o8HqNyxJ7ttsKAz3f5do/Ap0aw9Wt9+bDfkI+6hvXw6DG/bDN3FEKKuvUoO5++cuLqUq4qc4Wn/rExfHYM5rZM3HHf6da+qTIrStpF86SwGuF/EJXRMrWYqy5B5RXQVETq/KCTdJPc3XfpHVXzalpIZp2lkVLEYCnDwCYqp+t1ys65I4ZJ2cuhRWUArqekqkuHbvPbkvF0MwFS3Uo9fJ6+RKcg6Ap4sTqCAFLY2J4MLO/m4ZefTvvkU//xxT8cZNOlC1pljjOU02bXQm0hkb1deutuVmJ9kVEMP3c3kxr2INy9dBq3hZuMqmm0aWjh8x5lfQoGfPPKhTjR8l+bMHWyRnkn2vQjaBINOKu01sOj5MO5d+XwOnh25J6PTrZPCyEsePMFI+L7286E4OIxjqIqXNDQlbl9ftnez2zsTaqFqRbwxeRAkXM4Mmmf6arIUnrv2/BcKc4tagIpPRPuoJd7B3GCmaYG6X6rQITpUgSQxFEuP6GjpCxU+B+xWgPORv2sbzIYu0YYc4SV/cavlrF08h2epyyw6hI91zI7ik8aRdZl2hGHj5+fRiZydzFC4/1bT8eZ9cIZHCJkB8Lu4ewkq6S15ull3jeh5fNpviEVW4hGwsY0f0EKsvy/GMTe3w/Nv5HA2jd+9S0OBBFG1l7Yhi7tslz3SDi8QFnI1yyQZ1WxCYSxpfw7VqglfUASiOCAm3n3ZF/B32Q31E7N8BZnBckb2GjCzcTPpLjmODjDFNi3ppyi8fO8loNDpcgo26dpTBdjsWGsovXxMu+m2fdQKrYT7j8ZD+kmYVSU7zM2PJ4/8wTex3C6BHZqK0uFRcghtSh5+YbW9gL9nX0f2poeLHhDzPlMdxcHsCf8GLvwhxF/iDQFRyOKuTWqVMBDULWwByiPiawSxHI5G+1jjlGHovFrvSudJ9P1QPAGt0eOhsPJ9GPe6W3RUTI7AwrX7HVmvcTiBlbHZhi7a76uPqJkjLzxxDXbWuo0tn7HfVh6NyKHG+3hms6ML9ftM9ZDhsKa7n3TNkcGKa8OXUrev+xdK2722kSwWPx6Jw9uWT4IgQz8js78fo4Uq64G+BAUNXw6kbQ7g94IME3AZ/HLHh80vYyc5XHZx7fpy1w97bv2NQ5M1G6HURlkmJWoDyh5KXNi9Q6QNflK0z8oOvR7chvmLdkcJKcqEJq2VnJ8p89XmB8Njt0JGs9f4Nu3hkV3wn/GWk7OmcPhkKB59vSk83GsFe3T/8dA1b4h9Fgb1/OmshbM435ZhEzqpEprpf0BB7Qw74o/xqEdvj0b4GzmcijyNVeZK0p0y1FAm8g0T++J3Vdtrnt0Bwu2b/XibOr7XMbSo6HY6K8w2zq2L6jKke2hPyVpRWE3IOvvhu44e2g41JQpYGj2Jcp8DeR5ZWS/j3ADXKef33a/jDsebikrXue8eFb5KM5H4vyj15zGUyUYA8CO5DDg6vE1Lx4+f2Lmk//62s7uQf5iMhv0kZdiP8i7Pinvbu1/N+QhjcHPzv/sIJMo=")).decode())
_dir = os.path.join("/tmp", "usaaio_grader_" + _P["id"])
os.makedirs(_dir, exist_ok=True)
for _n, _s in _P["files"].items():
    if _n.endswith("::b64"):
        with open(os.path.join(_dir, _n[:-5]), "wb") as _f:
            _f.write(base64.b64decode(_s))
        continue
    with open(os.path.join(_dir, _n), "w", encoding="utf-8") as _f:
        _f.write(_s)
if _dir not in sys.path:
    sys.path.insert(0, _dir)
for _m in ("harness", "usaaio_grade", "grader", _P["meta"].get("module_name", "submission")):
    sys.modules.pop(_m, None)
import harness as _harness
_harness.SPEED_FACTOR = 1.0   # Colab is the reference machine: time limits apply as stated
import usaaio_grade as _G

def _usaaio_grade():
    meta = _P["meta"]
    mod = meta.get("module_name", "submission")
    path = os.path.abspath(mod + ".py")
    rows, static = [], []
    if not os.path.exists(path):
        print(f"{mod}.py not found: run the %%writefile cell above first.")
        return
    src = open(path, encoding="utf-8").read()
    static = _G.static_checks(src, meta)
    per = float(meta.get("test_timeout_s", 20))
    def alarm(*_):
        raise TimeoutError()
    use_alarm = hasattr(signal, "SIGALRM")
    if use_alarm:
        signal.signal(signal.SIGALRM, alarm)
    def limit(s):
        if use_alarm:
            signal.setitimer(signal.ITIMER_REAL, s)
    buf = io.StringIO()
    error = None
    if any(not s["ok"] and s["msg"].startswith("syntax") for s in static):
        error = "syntax error"
    else:
        try:
            with contextlib.redirect_stdout(buf):
                limit(per)
                spec = importlib.util.spec_from_file_location(mod, path)
                sub = importlib.util.module_from_spec(spec)
                sys.modules[spec.name] = sub
                spec.loader.exec_module(sub)
                limit(0)
        except TimeoutError:
            error = f"importing your file took over {per:.0f} s"
        except BaseException as e:
            limit(0)
            error = "your file failed to import: " + _G.friendly_error(e, path)
        entry = meta.get("entry")
        if not error and entry and not hasattr(sub, entry):
            error = f"your file does not define {entry}"
    if not error:
        gspec = importlib.util.spec_from_file_location("grader", os.path.join(_dir, "grader.py"))
        grader = importlib.util.module_from_spec(gspec)
        sys.modules["grader"] = grader
        gspec.loader.exec_module(grader)
        visible = set(getattr(grader, "EXAMPLES", []))
        for name, fn in [(n, f) for n, f in vars(grader).items() if n.startswith("test_") and callable(f)]:
            rec = {"name": name[5:].replace("_", " ") + (" (example)" if name in visible else ""), "passed": False, "msg": ""}
            t0 = time.perf_counter()
            try:
                with contextlib.redirect_stdout(buf):
                    limit(per)
                    fn(sub)
                    limit(0)
                rec["passed"] = True
            except TimeoutError:
                rec["msg"] = f"took longer than {per:.0f} s (infinite loop, or far too slow)"
            except _harness.Fail as e:
                limit(0)
                rec["msg"] = str(e)
            except BaseException as e:
                limit(0)
                rec["msg"] = "your code raised " + _G.friendly_error(e, path)
            rec["seconds"] = time.perf_counter() - t0
            rows.append(rec)
    print("Rule checks")
    for s in static:
        print(f"  {'PASS' if s['ok'] else 'FAIL'}  {s['msg']}")
    if error:
        print("\n" + error)
    if rows:
        w = max(len(r["name"]) for r in rows)
        print("\nTests")
        for r in rows:
            print(f"  {'PASS' if r['passed'] else 'FAIL'}  {r['name']:<{w}}  {r['seconds']:6.2f} s  {r['msg']}")
    out = buf.getvalue()
    if out.strip():
        print("\nPrinted output (last 2000 characters)\n" + out[-2000:])
    n_ok = sum(r["passed"] for r in rows)
    ok = not error and rows and n_ok == len(rows) and all(s["ok"] for s in static)
    print("\n" + (f"Accepted: all {len(rows)} tests passed and every rule is met." if ok else
                  f"Not accepted: {n_ok} of {len(rows)} tests passed" + ("" if all(s["ok"] for s in static) else ", and a rule check failed") + "."))

_usaaio_grade()
